In [ ]:
import os, json, random, time, warnings
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from scipy.stats import binomtest
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

DATA_DIR    = "/kaggle/input/datasets/sabbir4724/dentiro1"
GROUPS_CSV  = "/kaggle/working/groups.csv"
COL_PATH    = None
COL_FOLD    = "fold"
COL_GROUP   = "group"
PROPOSED_OOF = None

OUT_DIR     = "/kaggle/working/kfold_results_v2"
EPOCHS      = 40
N_FOLDS     = 5
INNER_SPLITS= 8
IMG_SIZE    = 256
CLAHE_CLIP  = 2.0
CLAHE_GRID  = 8
LABEL_SMOOTH= 0.1
BATCH_SIZE  = 32
LR          = 1e-4
WD          = 1e-2
SEED        = 42
NUM_WORKERS = 2
PRETRAINED  = True
DPI         = 600
MODELS      = ["resnet50", "resnet18", "mobilenetv3_large", "efficientnet_b0", "vit_b_16"]

CKPT_DIR = f"{OUT_DIR}/ckpt"; RES_DIR = f"{OUT_DIR}/folds"; PLOT_DIR = f"{OUT_DIR}/plots"; CSV_DIR = f"{OUT_DIR}/csv"
for d in [CKPT_DIR, RES_DIR, PLOT_DIR, CSV_DIR]: os.makedirs(d, exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base = datasets.ImageFolder(DATA_DIR)
CLASSES = base.classes; NUM_CLASSES = len(CLASSES)
df = pd.DataFrame({"path": [s[0] for s in base.samples], "label": [s[1] for s in base.samples]})
print(f"Images: {len(df)} | Classes ({NUM_CLASSES}): {CLASSES}")

gdf = pd.read_csv(GROUPS_CSV)
print("groups.csv columns:", gdf.columns.tolist(), "| rows:", len(gdf))

def _find(cols, cands):
    low = {c.lower(): c for c in cols}
    for k in cands:
        if k in low: return low[k]
    return None

cp = COL_PATH or _find(gdf.columns, ["path", "filepath", "file_path", "image_path", "image", "file", "filename", "img", "name"])
cg = COL_GROUP if COL_GROUP in gdf.columns else _find(gdf.columns, ["group", "groups", "patient_id", "patient", "group_id"])
cf = COL_FOLD  if COL_FOLD  in gdf.columns else _find(gdf.columns, ["fold", "test_fold", "kfold"])
assert cp is not None, f"Could not find a path column in groups.csv. Set COL_PATH. Columns: {gdf.columns.tolist()}"

def make_key(p, depth):
    return "/".join(str(p).replace("\\", "/").split("/")[-depth:])

merged = None
for depth in (2, 1):
    df["key"]  = df.path.map(lambda p: make_key(p, depth))
    gdf["key"] = gdf[cp].map(lambda p: make_key(p, depth))
    sel = ["key"] + [c for c in [cg, cf] if c is not None]
    m = df.merge(gdf[sel].drop_duplicates("key"), on="key", how="inner")
    if len(m) == len(df): merged = m; break
assert merged is not None, "Image paths do not match groups.csv (count mismatch). Check COL_PATH."
df = merged.reset_index(drop=True)

df["group"] = df[cg].values if cg is not None else np.arange(len(df))
if cf is None:
    assert cg is not None, "groups.csv has neither a fold nor a group column. Folds cannot be regenerated."
    print("WARNING: no fold column found, regenerating folds with StratifiedGroupKFold. They may NOT match the main notebook folds.")
    df["fold"] = -1
    for f, (_, te) in enumerate(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(df.path, df.label, df.group)):
        df.loc[te, "fold"] = f
else:
    df["fold"] = df[cf].astype(int).values
assert sorted(df.fold.unique()) == list(range(N_FOLDS)), f"unexpected folds: {df.fold.unique()}"

paths  = df.path.values
labels = df.label.values
groups = df.group.values

SPLITS = []
for f in range(N_FOLDS):
    te = np.where(df.fold.values == f)[0]; trv = np.where(df.fold.values != f)[0]
    if cg is not None: assert set(groups[te]).isdisjoint(set(groups[trv])), f"group leakage in fold {f}"
    sg = StratifiedGroupKFold(INNER_SPLITS, shuffle=True, random_state=SEED + f)
    a, b = next(sg.split(trv, labels[trv], groups[trv]))
    SPLITS.append((trv[a].tolist(), trv[b].tolist(), te.tolist()))
    print(f"Fold {f}: train {len(a)} | inner-val {len(b)} | test {len(te)}")
json.dump(SPLITS, open(f"{OUT_DIR}/splits.json", "w"))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def load_clahe(p):
    im = cv2.cvtColor(cv2.imread(p, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_GRID, CLAHE_GRID))
    lab = cv2.cvtColor(im, cv2.COLOR_RGB2LAB); lab[..., 0] = clahe.apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

class DS(Dataset):
    def __init__(self, p, l, tf): self.p, self.l, self.tf = p, l, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(load_clahe(self.p[i])), int(self.l[i])

class Resize224(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(F.interpolate(x, size=224, mode="bilinear", align_corners=False))

def build_model(name):
    w = "DEFAULT" if PRETRAINED else None
    if name == "resnet50":
        m = models.resnet50(weights=w); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "resnet18":
        m = models.resnet18(weights=w); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "mobilenetv3_large":
        m = models.mobilenet_v3_large(weights=w); m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=w); m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == "vit_b_16":
        try:
            from torchvision.models.vision_transformer import interpolate_embeddings
            m = models.vit_b_16(weights=None, image_size=IMG_SIZE)
            if PRETRAINED:
                sd = models.vit_b_16(weights="DEFAULT").state_dict()
                sd = interpolate_embeddings(image_size=IMG_SIZE, patch_size=16, model_state=sd, reset_heads=True)
                m.load_state_dict(sd, strict=False)
            m.heads.head = nn.Linear(m.heads.head.in_features, NUM_CLASSES)
        except ImportError:
            print("[ViT] interpolate_embeddings unavailable -> 224 internal resize fallback")
            m = models.vit_b_16(weights=w); m.heads.head = nn.Linear(m.heads.head.in_features, NUM_CLASSES)
            m = Resize224(m)
    elif name == "swin_tiny":
        m = models.swin_t(weights=w); m.head = nn.Linear(m.head.in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m.to(device)

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tl, yt, yp = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            o = model(x); loss = crit(o, y)
        tl += loss.item() * x.size(0)
        yt += y.cpu().tolist(); yp += o.argmax(1).cpu().tolist()
    return tl / len(yt), accuracy_score(yt, yp), f1_score(yt, yp, average="macro"), np.array(yt), np.array(yp)

def run_fold(name, fold):
    res_path = f"{RES_DIR}/{name}_fold{fold}.json"
    if os.path.exists(res_path): return json.load(open(res_path))
    tr_idx, va_idx, te_idx = [np.array(a) for a in SPLITS[fold]]
    mk = lambda idx, tf, bs, sh, dl: DataLoader(DS(paths[idx], labels[idx], tf), bs, shuffle=sh,
                                                num_workers=NUM_WORKERS, pin_memory=True, drop_last=dl)
    tr_ld = mk(tr_idx, train_tf, BATCH_SIZE, True, True)
    va_ld = mk(va_idx, val_tf, BATCH_SIZE * 2, False, False)
    te_ld = mk(te_idx, val_tf, BATCH_SIZE * 2, False, False)

    seed_all(SEED + fold)
    model = build_model(name)
    cnt = np.bincount(labels[tr_idx], minlength=NUM_CLASSES)
    cw  = torch.tensor(len(tr_idx) / (NUM_CLASSES * np.maximum(cnt, 1)), dtype=torch.float32, device=device)
    crit  = nn.CrossEntropyLoss(weight=cw, label_smoothing=LABEL_SMOOTH)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1"]}
    best_f1, best_ep, start = -1.0, -1, 0
    ck_path   = f"{CKPT_DIR}/{name}_fold{fold}_last.pth"
    best_path = f"{CKPT_DIR}/{name}_fold{fold}_best.pth"
    if os.path.exists(ck_path):
        ck = torch.load(ck_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        hist, best_f1, best_ep, start = ck["hist"], ck["best_f1"], ck["best_ep"], ck["epoch"] + 1
        print(f"[RESUME] {name} fold{fold} from epoch {start}")

    for ep in range(start, EPOCHS):
        model.train(); tl, correct, n = 0.0, 0, 0
        for x, y in tqdm(tr_ld, desc=f"{name} F{fold} E{ep+1}/{EPOCHS}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=device.type == "cuda"):
                o = model(x); loss = crit(o, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * x.size(0); correct += (o.argmax(1) == y).sum().item(); n += x.size(0)
        sched.step()
        vl, va, vf, _, _ = evaluate(model, va_ld, crit)
        for k, v in zip(hist.keys(), [tl / n, correct / n, vl, va, vf]): hist[k].append(v)
        if vf > best_f1:
            best_f1, best_ep = vf, ep + 1
            torch.save(model.state_dict(), best_path)
        print(f"{name} F{fold} E{ep+1:02d} | trL {tl/n:.4f} trA {correct/n:.4f} | vaL {vl:.4f} vaA {va:.4f} vaF1 {vf:.4f} | best@{best_ep}")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "hist": hist, "best_f1": best_f1, "best_ep": best_ep, "epoch": ep}, ck_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    _, ta, tf1, yt, yp = evaluate(model, te_ld, crit)
    rep = classification_report(yt, yp, target_names=CLASSES, output_dict=True, zero_division=0)
    out = {"hist": hist, "best_epoch": best_ep, "best_val_f1": best_f1, "test_acc": ta, "test_f1": tf1,
           "report": rep, "cm": confusion_matrix(yt, yp, labels=range(NUM_CLASSES)).tolist(),
           "te_idx": te_idx.tolist(), "y_true": yt.tolist(), "y_pred": yp.tolist(),
           "params_M": sum(p.numel() for p in model.parameters()) / 1e6}
    print(f"[TEST] {name} fold{fold}: acc {ta:.4f} | macroF1 {tf1:.4f} (ckpt epoch {best_ep})")
    json.dump(out, open(res_path, "w"))
    for p in [ck_path, best_path]:
        if os.path.exists(p): os.remove(p)
    del model, opt; torch.cuda.empty_cache()
    return out

def mean_std_plot(ax, arr, color, label=None):
    m, s = arr.mean(0), arr.std(0); x = np.arange(1, arr.shape[1] + 1)
    ax.plot(x, m, color=color, lw=2, label=label); ax.fill_between(x, m - s, m + s, color=color, alpha=0.25)

def report_model(name, results):
    cols = plt.cm.tab10(np.arange(N_FOLDS))
    H = {k: np.array([r["hist"][k] for r in results]) for k in results[0]["hist"]}
    ep = np.arange(1, EPOCHS + 1)
    specs = [("train_loss", "Training Loss (all folds)", "Loss", "all"),
             ("val_acc", "Inner-Val Accuracy (all folds)", "Accuracy", "all"),
             ("val_f1", "Inner-Val Macro-F1 (all folds)", "Macro-F1", "all"),
             ("train_loss", "Mean Training Loss ± Std", "Loss", "ms"),
             ("val_acc", "Mean Inner-Val Accuracy ± Std", "Accuracy", "ms"),
             ("val_f1", "Mean Inner-Val Macro-F1 ± Std", "Macro-F1", "ms")]
    def draw(ax, key, title, yl, mode):
        if mode == "all":
            for f in range(N_FOLDS): ax.plot(ep, H[key][f], color=cols[f], lw=1.5, label=f"Fold {f}")
        else:
            mean_std_plot(ax, H[key], "tab:blue", "mean ± std")
        ax.legend(fontsize=8); ax.set_title(f"{name}: {title}", fontsize=10)
        ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
    fig, axs = plt.subplots(2, 3, figsize=(18, 9))
    for ax, sp in zip(axs.ravel(), specs): draw(ax, *sp)
    plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/{name}_combined.png", dpi=DPI, bbox_inches="tight"); plt.show()
    for i, sp in enumerate(specs):
        fig, ax = plt.subplots(figsize=(6, 4.5)); draw(ax, *sp)
        plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/{name}_{i+1}_{sp[0]}_{sp[3]}.png", dpi=DPI, bbox_inches="tight"); plt.close()

    rows = []
    for f, r in enumerate(results):
        for c in CLASSES:
            rep = r["report"][c]
            rows.append({"model": name, "fold": f, "best_epoch": r["best_epoch"], "class": c,
                         "precision": rep["precision"], "recall": rep["recall"], "f1": rep["f1-score"], "support": rep["support"]})
    pdf = pd.DataFrame(rows); pdf.to_csv(f"{CSV_DIR}/{name}_perclass_foldwise.csv", index=False)
    print(f"\n===== {name}: TEST fold-wise per-class =====")
    for f in range(N_FOLDS):
        print(f"\n-- Fold {f} (ckpt epoch {results[f]['best_epoch']}) --")
        print(pdf[pdf.fold == f][["class", "precision", "recall", "f1", "support"]].round(4).to_string(index=False))
    pc = pdf.groupby("class")[["precision", "recall", "f1"]].agg(["mean", "std"]).round(4)
    pc.to_csv(f"{CSV_DIR}/{name}_perclass_mean_std.csv")
    print(f"\n-- {name}: per-class mean ± std over folds (TEST) --"); print(pc)

    summ = pd.DataFrame([{"model": name, "fold": f, "best_epoch": r["best_epoch"], "inner_val_f1": r["best_val_f1"],
                          "test_acc": r["test_acc"], "test_macro_f1": r["test_f1"], "params_M": r["params_M"]}
                         for f, r in enumerate(results)])
    summ.to_csv(f"{CSV_DIR}/{name}_fold_summary.csv", index=False)
    print(f"\n-- {name}: fold summary --"); print(summ.round(4).to_string(index=False))
    print(f"TEST Acc {summ.test_acc.mean():.4f} ± {summ.test_acc.std():.4f} | "
          f"Macro-F1 {summ.test_macro_f1.mean():.4f} ± {summ.test_macro_f1.std():.4f}")
    return H, summ

ALL_H, ALL_SUMM, ALL_RES = {}, [], {}
for name in MODELS:
    t0 = time.time(); results = [run_fold(name, f) for f in range(N_FOLDS)]
    H, summ = report_model(name, results)
    ALL_H[name] = H; ALL_SUMM.append(summ); ALL_RES[name] = results
    print(f"[DONE] {name} in {(time.time()-t0)/60:.1f} min")

final = pd.concat(ALL_SUMM)
tab = final.groupby("model").agg(acc_mean=("test_acc", "mean"), acc_std=("test_acc", "std"),
                                 f1_mean=("test_macro_f1", "mean"), f1_std=("test_macro_f1", "std"),
                                 params_M=("params_M", "first")).round(4)
tab.to_csv(f"{CSV_DIR}/all_models_summary.csv"); print("\n===== ALL MODELS (TEST, mean over 5 folds) =====\n", tab)

pal = plt.cm.tab10(np.arange(len(MODELS)))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (key, t) in zip(axs, [("train_loss", "Mean Training Loss ± Std"),
                              ("val_acc", "Mean Inner-Val Accuracy ± Std"),
                              ("val_f1", "Mean Inner-Val Macro-F1 ± Std")]):
    for i, n in enumerate(ALL_H): mean_std_plot(ax, ALL_H[n][key], pal[i], n)
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/all_models_mean_std.png", dpi=DPI, bbox_inches="tight"); plt.show()

N = len(df); oof = pd.DataFrame({"idx": np.arange(N), "path": paths, "fold": df.fold.values, "y_true": labels})
for name in MODELS:
    p = np.full(N, -1)
    for r in ALL_RES[name]: p[r["te_idx"]] = r["y_pred"]
    assert (p >= 0).all(); oof[f"pred_{name}"] = p
oof.to_csv(f"{CSV_DIR}/oof_predictions.csv", index=False)
print("\n===== POOLED OOF (all images) =====")
for name in MODELS:
    pr = oof[f"pred_{name}"].values
    print(f"{name:20s} acc {accuracy_score(labels, pr):.4f} | macroF1 {f1_score(labels, pr, average='macro'):.4f}")

if PROPOSED_OOF and os.path.exists(PROPOSED_OOF):
    po = pd.read_csv(PROPOSED_OOF)
    if "idx" in po.columns: prop = po.sort_values("idx").y_pred.values
    else:
        pc_ = _find(po.columns, ["path", "filepath", "image", "filename"])
        for depth in (2, 1):
            ks = po[pc_].map(lambda p: make_key(p, depth)); kd = df.path.map(lambda p: make_key(p, depth))
            mp = dict(zip(ks, po.y_pred))
            if kd.isin(mp).all(): prop = kd.map(mp).values; break
        else: raise RuntimeError("PROPOSED_OOF paths do not match the dataset images")
    assert len(prop) == N
    pc = prop == labels; rows = []
    for name in MODELS:
        bc = oof[f"pred_{name}"].values == labels
        b = int((pc & ~bc).sum()); c = int((~pc & bc).sum())
        p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else 1.0
        rows.append({"baseline": name, "b_prop_only": b, "c_base_only": c, "p_raw": p})
    mc = pd.DataFrame(rows).sort_values("p_raw").reset_index(drop=True)
    m_ = len(mc); mc["p_holm"] = np.minimum(1.0, np.maximum.accumulate((m_ - np.arange(m_)) * mc.p_raw.values))
    mc.to_csv(f"{CSV_DIR}/mcnemar_vs_proposed.csv", index=False)
    print("\n===== McNemar (proposed vs baselines, pooled OOF) =====\n", mc.round(6).to_string(index=False))
else:
    print("\n[McNemar skipped] Set PROPOSED_OOF (csv with path or idx, plus y_pred, from the DentAttn-MSFNet pooled OOF). "
          f"Baseline OOF saved: {CSV_DIR}/oof_predictions.csv")

In [ ]:
# ===== Summary figure: all folds, all models =====
import os, json
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR  = "/kaggle/working/kfold_results"
RES_DIR  = f"{OUT_DIR}/folds"
PLOT_DIR = f"{OUT_DIR}/plots"
os.makedirs(PLOT_DIR, exist_ok=True)

MODELS  = ["resnet50", "resnet18", "mobilenetv3_large", "efficientnet_b0", "vit_b_16"]
N_FOLDS = 5
DPI     = 1000

METRICS = [("train_loss", "Training Loss",            "Loss"),
           ("val_acc",    "Validation Accuracy",      "Accuracy"),
           ("val_f1",     "Validation Macro-F1",      "Macro-F1")]

# ---- load histories ----
H = {}
for m in MODELS:
    runs = [json.load(open(f"{RES_DIR}/{m}_fold{f}.json"))["hist"] for f in range(N_FOLDS)]
    H[m] = {k: np.array([r[k] for r in runs]) for k in runs[0]}   # (folds, epochs)

cols = plt.cm.tab10(np.arange(N_FOLDS))

# ---- Figure 1: all folds, all models ----
fig, axs = plt.subplots(len(MODELS), 3, figsize=(18, 3.6 * len(MODELS)))
for i, m in enumerate(MODELS):
    for j, (key, title, yl) in enumerate(METRICS):
        ax = axs[i, j]; arr = H[m][key]; ep = np.arange(1, arr.shape[1] + 1)
        for f in range(N_FOLDS):
            ax.plot(ep, arr[f], color=cols[f], lw=1.4, label=f"Fold {f}")
        ax.set_title(f"{m}: {title}", fontsize=10)
        ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
        if i == 0 and j == 0: ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/summary_allfolds_all_models.png", dpi=DPI, bbox_inches="tight")
plt.show()

# ---- Figure 2: model-wise mean ± std (3 panel, all models in one plot) ----
pal = plt.cm.tab10(np.arange(len(MODELS)))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (key, title, yl) in zip(axs, METRICS):
    for i, m in enumerate(MODELS):
        arr = H[m][key]; mu, sd = arr.mean(0), arr.std(0); ep = np.arange(1, arr.shape[1] + 1)
        ax.plot(ep, mu, color=pal[i], lw=2, label=m)
        ax.fill_between(ep, mu - sd, mu + sd, color=pal[i], alpha=0.2)
    ax.set_title(f"Mean {title} ± Std"); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/summary_mean_std_all_models.png", dpi=DPI, bbox_inches="tight")
plt.show()